# Sanity Check — agent_features

Verifica qualità del dataset prima di procedere con ML:
1. Copertura (NULL per colonna)
2. Distribuzioni delle feature principali
3. Correlazioni
4. Outlier

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

conn = sqlite3.connect("../data/moltbook.db")
df = pd.read_sql("SELECT * FROM agent_features", conn)

print(f"Righe totali: {len(df)}")
print(f"Colonne: {len(df.columns)}")

## 1. Copertura — NULL per colonna

In [ ]:
# Conta NULL e percentuale per ogni colonna
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

coverage = pd.DataFrame({
    "non_null": len(df) - null_counts,
    "null": null_counts,
    "null_%": null_pct
}).sort_values("null_%", ascending=False)

print(coverage.to_string())

In [ ]:
# Visualizzazione copertura
fig, ax = plt.subplots(figsize=(10, 8))
coverage["null_%"].plot(kind="barh", ax=ax, color="tomato")
ax.set_xlabel("% NULL")
ax.set_title("Percentuale di valori NULL per feature")
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Focus sul sottoinsieme utile (agenti nel grafo)

Filtriamo sugli agenti che hanno `in_degree IS NOT NULL` — quelli effettivamente nel grafo conversazionale.

In [ ]:
# Agenti nel grafo vs fuori dal grafo
df_graph = df[df["in_degree"].notna()].copy()
df_no_graph = df[df["in_degree"].isna()].copy()

print(f"Agenti nel grafo:       {len(df_graph)}")
print(f"Agenti fuori dal grafo: {len(df_no_graph)}")
print(f"\nCopertura NULL nel sottoinsieme GRAFO:")
null_graph = df_graph.isnull().sum()
null_graph_pct = (null_graph / len(df_graph) * 100).round(1)
print(null_graph_pct[null_graph_pct > 0].sort_values(ascending=False).to_string())

## 3. Statistiche descrittive

In [ ]:
# Statistiche sul subset del grafo
feature_cols = [
    "n_posts", "n_comments", "n_comments_received",
    "active_days", "burstiness_posts", "hour_entropy",
    "reply_to_post_ratio", "self_reply_rate", "unique_targets", "mean_thread_depth",
    "mean_post_length", "std_post_length", "type_token_ratio",
    "in_degree", "out_degree", "pagerank", "betweenness",
    "local_clustering", "egonet_size", "egonet_density", "reciprocity_local"
]

print(df_graph[feature_cols].describe().round(4).to_string())

## 4. Distribuzioni delle feature principali

In [ ]:
# Istogrammi delle feature chiave (scala log dove serve)
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

plot_features = [
    ("n_posts", True),
    ("n_comments", True),
    ("n_comments_received", True),
    ("active_days", False),
    ("burstiness_posts", False),
    ("hour_entropy", False),
    ("reply_to_post_ratio", True),
    ("self_reply_rate", False),
    ("unique_targets", True),
    ("mean_thread_depth", False),
    ("type_token_ratio", False),
    ("in_degree", True),
    ("out_degree", True),
    ("pagerank", True),
    ("betweenness", True),
    ("local_clustering", False),
]

for i, (feat, use_log) in enumerate(plot_features):
    ax = axes[i]
    data = df_graph[feat].dropna()
    ax.hist(data, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(feat, fontsize=10)
    if use_log:
        ax.set_yscale("log")

plt.suptitle("Distribuzioni feature — agenti nel grafo", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlazioni

In [ ]:
# Matrice di correlazione sulle feature numeriche
corr = df_graph[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Matrice di correlazione — feature agenti nel grafo")
plt.tight_layout()
plt.show()

In [ ]:
# Top correlazioni (escludendo diagonale e duplicati)
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_stack = corr_pairs.stack().reset_index()
corr_stack.columns = ["feature_1", "feature_2", "correlation"]
corr_stack["abs_corr"] = corr_stack["correlation"].abs()

print("=== TOP 15 CORRELAZIONI ===")
print(corr_stack.sort_values("abs_corr", ascending=False).head(15)[
    ["feature_1", "feature_2", "correlation"]
].to_string(index=False))

## 6. Confronto claimed vs unclaimed

In [ ]:
# Statistiche per gruppo
claimed = df_graph[df_graph["is_claimed"] == 1]
unclaimed = df_graph[df_graph["is_claimed"] == 0]

print(f"Nel grafo — Claimed: {len(claimed)}, Unclaimed: {len(unclaimed)}")
print(f"\n{'Feature':<22} {'Claimed (median)':>16} {'Unclaimed (median)':>18}")
print("-" * 58)

for feat in feature_cols:
    med_c = claimed[feat].median()
    med_u = unclaimed[feat].median()
    if pd.notna(med_c) or pd.notna(med_u):
        print(f"{feat:<22} {med_c:>16.4f} {med_u:>18.4f}")

## 7. Outlier — agenti estremi

In [ ]:
# Top 10 per out_degree (i più attivi nel rispondere)
agents_names = pd.read_sql("SELECT id, name FROM agents", conn)
df_graph_named = df_graph.merge(agents_names, left_on="agent_id", right_on="id", how="left")

print("=== TOP 10 OUT-DEGREE ===")
print(df_graph_named.nlargest(10, "out_degree")[["name", "out_degree", "in_degree", "pagerank", "is_claimed"]].to_string(index=False))

print("\n=== TOP 10 PAGERANK ===")
print(df_graph_named.nlargest(10, "pagerank")[["name", "pagerank", "in_degree", "out_degree", "is_claimed"]].to_string(index=False))

print("\n=== TOP 10 BETWEENNESS ===")
print(df_graph_named.nlargest(10, "betweenness")[["name", "betweenness", "in_degree", "out_degree", "is_claimed"]].to_string(index=False))

In [ ]:
conn.close()
print("Done.")